In [ ]:
%load_ext autoreload

import os
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import numpy as np

import mcfs
import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
# %matplotlib widget
%matplotlib inline

# Load fake_spectra lines

In [ ]:
base_dir = "/home/STORAGE"
sim_name = "TNG100-1"
snap_num = 33
delta_grid = [0.5, 2.0, 10.0, 25.0]
res_kms = 1.0

data = mcfs.load_runs.load_data(base_dir=base_dir, sim_name=sim_name, snap_num=snap_num, delta_grid=delta_grid, res_kms=res_kms, preset="lya_si")

sweep_list = delta_grid
fiducial_key = sweep_list[0]

### postproc tau

In [ ]:
_tau_factor = 1e0
array_tau_factors = np.array([1.0, _tau_factor, None, None], dtype=object) # One entry per line in tau_loaded. Use None for lines you want to discard

In [ ]:
keep_mask = np.array([f is not None for f in array_tau_factors], dtype=bool) # Keep only entries with a valid scaling factor
tau_factors_kept = np.array(array_tau_factors[keep_mask], dtype=float) # Convert kept factors to float

In [ ]:
tau_scaled = {}
v_kms = {}
for key_case in sweep_list:
    tau_loaded = data["data"][key_case]["tau"]
    tau_scaled[key_case] = tau_loaded[keep_mask] * tau_factors_kept[:, None, None] # Filter tau_loaded and apply the corresponding scaling
    v_kms[key_case] = data["data"][key_case]["v_kms"]

# overflux

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA") # "SiII-1190", "SiII-1193"

In [ ]:
tau_cube = {}
tau_shifted = {}
overflux = {}
overflux_total = {}
overflux_subsets = {}
subset_labels = {}

P1D = {}
omega_tot = {}
P1D_tot = {}
P1D_tot_mean = {}
P1D_tot_std = {}
omega = {}
P1D_catalog = {}
avg_P1D_catalog = {}
by_order = {}

Xi1D = {}
Xi1D_tot = {}
Xi1D_tot_mean = {}
Xi1D_tot_std = {}
lags_tot = {}
lags = {}
Xi1D_catalog = {}
avg_Xi1D_catalog = {}
by_order_xi = {}

for key_case in sweep_list:
    tau_cube[key_case] = mcfs.overflux_tools.OpticalDepthCube(tau=tau_scaled[key_case], line_bundle=line_bundle)
    tau_shifted[key_case] = tau_cube[key_case].periodic_shift(x=v_kms[key_case], period=np.max(v_kms[key_case]))
    overflux[key_case], overflux_total[key_case] = tau_cube[key_case].build_overflux(tau=tau_shifted[key_case], ens_axis=0)
    overflux_subsets[key_case], subset_labels[key_case] = tau_cube[key_case].build_subset_fields(overflux=overflux[key_case], max_order=None)

    P1D[key_case] = mcfs.P1D.P1DAnalyzer(lambda_or_v=v_kms[key_case], optical_depth_cube=tau_cube[key_case])
    omega_tot[key_case], P1D_tot[key_case] = P1D[key_case].compute_total_P1D(
        overflux_total=overflux_total[key_case], axis=-1, subtract_mean=False, drop_zero_mode=True
    )
    P1D_tot_mean[key_case] = np.mean(P1D_tot[key_case], axis=0)
    omega[key_case], P1D_catalog[key_case], by_order[key_case] = P1D[key_case].compute_subset_P1D_catalog(
        subset_fields=overflux_subsets[key_case], subset_labels=subset_labels[key_case], axis=-1, subtract_mean=False, drop_zero_mode=True,
    )
    omega[key_case], avg_P1D_catalog[key_case] = P1D[key_case].compute_average_P1D_catalog( # Average over skewers
        P1D_catalog=P1D_catalog[key_case], average_axis=0, return_std=False, return_sem=True, symmetrize=True
    )

    Xi1D[key_case] = mcfs.Xi1D.Xi1DAnalyzer(lambda_or_v=v_kms[key_case], optical_depth_cube=tau_cube[key_case])
    lags_tot[key_case], Xi1D_tot[key_case] = Xi1D[key_case].compute_total_Xi1D_from_P1D(P1D_total=P1D_tot[key_case], axis=-1, P1D_zero_mode_dropped=True, center_lags=False)
    Xi1D_tot_mean[key_case] = np.mean(Xi1D_tot[key_case], axis=0)
    lags[key_case], Xi1D_catalog[key_case], by_order_xi[key_case] = Xi1D[key_case].compute_subset_Xi1D_catalog_from_P1D(
        P1D_catalog=P1D_catalog[key_case], axis=-1, P1D_zero_mode_dropped=True, center_lags=False
    )
    lags[key_case], avg_Xi1D_catalog[key_case] = Xi1D[key_case].compute_average_Xi1D_catalog(
        Xi1D_catalog=Xi1D_catalog[key_case], average_axis=0, return_std=False, return_sem=True, symmetrize=True,
    )

# Plots

In [ ]:
n_plot = 1               # number of random skewers to display
seed = 1234              # for reproducibility

# Random skewer selection
rng = np.random.default_rng(seed)
ii_list =[0]

In [ ]:
figs_tau = mcfs.plotting_utils.plot_tau_flux_overflux_resolution_comparison(
    tau_cube=tau_cube, tau_plot=tau_shifted,          # or another tau-like dict
    overflux=overflux, overflux_total=overflux_total, v_kms=v_kms,
    ii_list=ii_list, case_order=sweep_list,   # first = fiducial / best resolved
    fiducial_key=fiducial_key, line_ids=None,                 # or e.g. ["HI-LyA", "SiIII-1206"]
    plot_total=True, figsize=(16, 18), cmin=0.30, cmax=1.0,
)

# P1D

In [ ]:
figs_p1d = mcfs.plotting_utils.plot_p1d_resolution_comparison(
    avg_P1D_catalog=avg_P1D_catalog, omega=omega, P1D_tot_mean=P1D_tot_mean, omega_tot=omega_tot,
    case_order=sweep_list, fiducial_key=fiducial_key, figsize=(12, 8), cmin=0.30, cmax=1.0, max_x=0.045,
    residual_ylim=(-0.05, 0.05), residual_band=0.01, residual_relative_floor=1e-14,
    xscale="linear", yscale="log", y_min=0.007, y_max=6e-1
)

# $\xi1D$

In [ ]:
figs_xi1d = mcfs.plotting_utils.plot_xi1d_resolution_comparison(
    avg_Xi1D_catalog=avg_Xi1D_catalog, lags=lags, Xi1D_tot_mean=Xi1D_tot_mean, lags_tot=lags_tot, line_bundle=line_bundle,
    case_order=sweep_list, fiducial_key=fiducial_key, figsize=(12, 8), cmin=0.30, cmax=1.0, positive_lags_only=True,
    show_line_pair_markers=True, residual_ylim=(-0.1, 0.1), residual_band=0.01, residual_relative_floor=1e-14,
    xscale="linear", yscale="linear", # y_min=-0.02, y_max=0.22, max_x=2500.0,
)